In [ ]:
import joblib

from jpeg_compression_detection.paths import (
    DATA_DIR,
    GENERATED_DATA_DIR,
    MODELS_DIR,
    FIGURES_DIR,
    display_path,
)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load every saved compression-quality model except the stable best-model alias.
model_files = list(MODELS_DIR.glob('*_compression_model.pkl'))
loaded_results = {}

print("Loading saved models...")
print("=" * 70)

for model_path in model_files:
    model_name = model_path.name.replace('_compression_model.pkl', '').replace('_', ' ').title()
    if model_name == 'Best':
        continue

    model = joblib.load(model_path)
    loaded_results[model_name] = {'model': model}

    print(f"Loaded: {model_name}")
    print(f"  File: {model_path.name}")
    print(f"  Type: {type(model).__name__}")
    print()

print(f"\nLoaded {len(loaded_results)} models")
print("=" * 70)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

FONT_SIZE = 16

# Load the JFIF feature table.
data_path = DATA_DIR / 'results_jfif.csv'
from_csv = pd.read_csv(data_path).dropna()
X_from_csv = from_csv.drop(['filename', 'extension', 'estimated_quality'], axis=1)

print(f"JFIF data loaded: {len(X_from_csv)} samples")
print("=" * 70)

y_pred_jfif_loaded = {}
for model_name, result in loaded_results.items():
    y_pred_jfif_loaded[model_name] = result['model'].predict(X_from_csv)
    print(f"Prediction completed: {model_name}")

print("\n" + "=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for index, (model_name, y_pred) in enumerate(y_pred_jfif_loaded.items()):
    axes[index].hist(y_pred, bins=50, alpha=0.7, edgecolor='black', color='skyblue')
    axes[index].axvline(
        x=y_pred.mean(), color='r', linestyle='--', lw=2,
        label=f'Mean = {y_pred.mean():.2f}',
    )
    axes[index].axvline(
        x=np.median(y_pred), color='g', linestyle='--', lw=2,
        label=f'Median = {np.median(y_pred):.2f}',
    )
    axes[index].set_xlabel('Predicted image quality', fontsize=FONT_SIZE)
    axes[index].set_ylabel('Number of images', fontsize=FONT_SIZE)
    axes[index].set_title(
        f'{model_name}\nMin={y_pred.min():.1f}, Max={y_pred.max():.1f}, Std={y_pred.std():.2f}',
        fontsize=FONT_SIZE,
    )
    axes[index].grid(True, alpha=0.3)
    axes[index].legend(fontsize=FONT_SIZE)
    axes[index].set_xlim(0, 100)
    axes[index].tick_params(axis='both', which='major', labelsize=FONT_SIZE)

plt.tight_layout()
figure_path = FIGURES_DIR / 'jfif_quality_predictions_loaded_models.png'
plt.savefig(figure_path, dpi=150, bbox_inches='tight')
plt.show()

print("\nQuality-prediction statistics for JFIF images:")
print("=" * 70)
for model_name, y_pred in y_pred_jfif_loaded.items():
    print(f"\n{model_name}:")
    print(f"  Mean: {y_pred.mean():.2f}")
    print(f"  Median: {np.median(y_pred):.2f}")
    print(f"  Std. dev.: {y_pred.std():.2f}")
    print(f"  Min: {y_pred.min():.2f}")
    print(f"  Max: {y_pred.max():.2f}")
    print(f"  Q1 (25%): {np.percentile(y_pred, 25):.2f}")
    print(f"  Q3 (75%): {np.percentile(y_pred, 75):.2f}")


# Prediction with one model


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Evaluate the Random Forest model on the PPM feature table.
data_path = DATA_DIR / 'results_ppm.csv'
from_csv = pd.read_csv(data_path).dropna()
X_from_csv = from_csv.drop(['filename', 'extension', 'estimated_quality'], axis=1)

print(f"PPM data loaded: {len(X_from_csv)} samples")
print("=" * 70)

model_name = 'Random Forest'
model = loaded_results[model_name]['model']
y_pred = model.predict(X_from_csv)
print(f"Prediction completed: {model_name}")

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(y_pred, bins=50, alpha=0.7, edgecolor='black', color='skyblue')
ax.axvline(x=y_pred.mean(), color='r', linestyle='--', lw=2,
           label=f'Mean = {y_pred.mean():.2f}')
ax.axvline(x=np.median(y_pred), color='g', linestyle='--', lw=2,
           label=f'Median = {np.median(y_pred):.2f}')
ax.set_xlabel('Predicted image quality', fontsize=FONT_SIZE)
ax.set_ylabel('Number of images', fontsize=FONT_SIZE)
ax.set_title(
    f'{model_name}\nMin={y_pred.min():.1f}, Max={y_pred.max():.1f}, Std={y_pred.std():.2f}',
    fontsize=FONT_SIZE,
)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=FONT_SIZE)
ax.set_xlim(0, 100)
ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)

plt.tight_layout()
figure_path = FIGURES_DIR / 'ppm_quality_predictions_single_model.png'
plt.savefig(figure_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\nQuality-prediction statistics for PPM images ({model_name}):")
print("=" * 70)
print(f"  Mean: {y_pred.mean():.2f}")
print(f"  Median: {np.median(y_pred):.2f}")
print(f"  Std. dev.: {y_pred.std():.2f}")
print(f"  Min: {y_pred.min():.2f}")
print(f"  Max: {y_pred.max():.2f}")
print(f"  Q1 (25%): {np.percentile(y_pred, 25):.2f}")
print(f"  Q3 (75%): {np.percentile(y_pred, 75):.2f}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# This external evaluation table is optional and is not distributed with the repository.
optional_data_path = GENERATED_DATA_DIR / 'results_ECVP_600.csv'

if not optional_data_path.exists():
    print(f"Skipping optional dataset because it is not available: {display_path(optional_data_path)}")
else:
    from_csv = pd.read_csv(optional_data_path).dropna()
    X_from_csv = from_csv.drop(['filename', 'extension', 'estimated_quality'], axis=1)

    print(f"Optional dataset loaded: {len(X_from_csv)} samples")
    print("=" * 70)

    model_name = 'Random Forest'
    model = loaded_results[model_name]['model']
    y_pred = model.predict(X_from_csv)
    print(f"Prediction completed: {model_name}")

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(y_pred, bins=50, alpha=0.7, edgecolor='black', color='skyblue')
    ax.axvline(x=y_pred.mean(), color='r', linestyle='--', lw=2,
               label=f'Mean = {y_pred.mean():.2f}')
    ax.axvline(x=np.median(y_pred), color='g', linestyle='--', lw=2,
               label=f'Median = {np.median(y_pred):.2f}')
    ax.set_xlabel('Predicted image quality', fontsize=FONT_SIZE)
    ax.set_ylabel('Number of images', fontsize=FONT_SIZE)
    ax.set_title(
        f'{model_name}\nMin={y_pred.min():.1f}, Max={y_pred.max():.1f}, Std={y_pred.std():.2f}',
        fontsize=FONT_SIZE,
    )
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=FONT_SIZE)
    ax.set_xlim(0, 100)
    ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)

    plt.tight_layout()
    figure_path = FIGURES_DIR / 'ecvp_quality_predictions_single_model.png'
    plt.savefig(figure_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\nQuality-prediction statistics ({model_name}):")
    print("=" * 70)
    print(f"  Mean: {y_pred.mean():.2f}")
    print(f"  Median: {np.median(y_pred):.2f}")
    print(f"  Std. dev.: {y_pred.std():.2f}")
    print(f"  Min: {y_pred.min():.2f}")
    print(f"  Max: {y_pred.max():.2f}")
    print(f"  Q1 (25%): {np.percentile(y_pred, 25):.2f}")
    print(f"  Q3 (75%): {np.percentile(y_pred, 75):.2f}")


# Single-image live prediction


In [28]:
from jpeg_compression_detection import detect_jpeg_compression

def check_image(image_path):
    """Extract indicators and predict JPEG quality for one image."""
    if not image_path.exists():
        print(f"Skipping optional example because it is not available: {display_path(image_path)}")
        return

    print(f"Analyzing file: {display_path(image_path)}")
    result = detect_jpeg_compression(image_path)
    if not isinstance(result, dict):
        print("The image indicators could not be computed.")
        return

    loaded_feature_columns = [
        'blockiness_0', 'blockiness_1', 'blockiness_2', 'blockiness_3',
        'blockiness_4', 'blockiness_5', 'blockiness_6', 'blockiness_7',
        'blockiness_max', 'blockiness_mean', 'blockiness_std',
        'energy_mean', 'energy_std', 'psnr',
    ]
    feature_dict = {column: result.get(column, 0) for column in loaded_feature_columns}
    X_single = pd.DataFrame([feature_dict])

    if X_single.isnull().any().any():
        print("\nMissing values in the indicators:")
        print(X_single.isnull().sum()[X_single.isnull().sum() > 0])
        return

    model_name = 'Random Forest'
    model = loaded_results[model_name]['model']
    y_pred = model.predict(X_single)

    print(f"\nPredicted image quality ({model_name}): {y_pred[0]:.2f}")
    print("-" * 80)


In [ ]:
example_paths = [
    DATA_DIR / 'examples' / 'fakeface.jpg',
    DATA_DIR / 'examples' / 'fakeface-100x100.png',
    DATA_DIR / 'examples' / 'fakeface-500x500.png',
    DATA_DIR / 'examples' / 'fakeface-250x250.png',
    DATA_DIR / 'examples' / 'test-500-700q60q100-1000x1500.png',
]

for example_path in example_paths:
    check_image(example_path)
